In [1]:
import json
import pandas as pd
import re

import requests
import zipfile
import io
# Extract the contents of the zip file to a temporary directory
import tempfile
import os
import shutil

In [2]:
# Download the zip file from the URL
temp_dir = tempfile.mkdtemp()

def download_unzip_files(temp_dir):
    url = 'https://owncloud.gwdg.de/index.php/s/wqUJmN53BYZK9ff/download'
    response = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(response.content))

    
    z.extractall(temp_dir)

def delete_temporary_files(temp_dir):
    shutil.rmtree(temp_dir)


In [4]:
# Opening JSON file

def remove_blanks_concat(txt_list):
    clean_list = [x for x in txt_list if x!="_BLANK"]
    concat = " ".join(clean_list)
    
    return concat

def remove_double_spaces(txt):
    clean_txt = re.sub(' +', ' ', txt).strip()
    return clean_txt


# Process json files
download_unzip_files(temp_dir)


dict_data={}

for root, dirs, files in os.walk(temp_dir):
    for filename in files:
        # Do something with each file
        file_path = os.path.join(root, filename)
        if "json" in filename:
            category= filename.split("_")[1]
            f = open(file_path)
            data = json.load(f)
            dict_data[category] = data
        # with open(file_path, 'r') as f:
        #     contents = f.read()
        #     print(filename)

delete_temporary_files(temp_dir)


# turn into table
list_dfs= []
for k,v in dict_data.items():
    df= pd.DataFrame.from_dict(v , orient='index')
    df["split"] = k
    list_dfs.append(df)


df = pd.concat(list_dfs)
df["text"]= df["text"].apply(remove_blanks_concat)
df["text"]= df["text"].apply(remove_double_spaces)
df["headline"]= df["headline"].apply(remove_double_spaces)

In [6]:
df.query("headline!=text").head()

,mid,midid,incidentid,headline,text,date,split
19209NYT,False,NaN,19209NYTIncident,Lord Sherfield Is Dead at 92; Was British Envo...,Lord Sherfield Is Dead at 92 Was British Envoy...,1996-11-11,train
18785NYT,False,NaN,18785NYTIncident,Paris Blames U.S. Position For Setback Over NATO,Paris Blames U S Position For Setback Over NATO,1996-10-13,train
11420NYT,False,NaN,11420NYTIncident,U.S. AND CHINESE RESOLVE DISPUTE ON MISSILE SALES,U S AND CHINESE RESOLVE DISPUTE ON MISSILE SALES,1994-10-05,train
12282NYT,False,NaN,12282NYTIncident,World News Briefs; Italian Prime Minister Conf...,World News Briefs Italian Prime Minister Confr...,1995-03-04,train
11560NYT,False,NaN,11560NYTIncident,World News Briefs; European Unity Policy Split...,World News Briefs European Unity Policy Splits...,1994-10-12,train


In [ ]:
# Will use headline column to keep punctuation